In [1]:
from Models_node import *
from utils.datautils import *
from utils.train_utils import *
import pickle
import sys
import time
import os

In [3]:
Dataset_name = "KentuckyDecisionTreeData"
# Max_epoch = 2
# print("Start Training ---" + str(Dataset_name) + " ---dataset\n")
dataset_path_ = "../UCRArchive_2018/"
normalize_dataset = True
# model training
# Xtrain_raw, ytrain_raw, Xval_raw, yval_raw, Xtest_raw, ytest_raw = Readdataset(dataset_path_, Dataset_name)
# Xtrain, Xval, Xtest = Multi_view(Xtrain_raw, Xval_raw, Xtest_raw)
start_time = time.time()
Xtrain, ytrain, Xval, yval, Xtest, ytest = Readdataset(dataset_path_, Dataset_name, standalize = False)
N, T = calculate_dataset_metrics(Xtrain)
end_time = time.time()
# export_file_path = f"../Preprocessed_data/{Dataset_name}_data.pkl"
# Create directory if it doesn't exist
# directory = os.path.dirname(export_file_path)
# os.makedirs(directory, exist_ok=True)
# with open(export_file_path, "wb") as data_file:
#     pickle.dump((Xtrain, ytrain, Xval, yval, Xtest, ytest, N, T), data_file)
# print(f"Preprocessed data saved to {export_file_path}")
print("Preprocessing time: {:.2f} seconds".format(end_time - start_time))

Preprocessing time: 0.08 seconds


In [5]:
import os
from Models_node import *

# Define the path to the tree model
tree_model_dir = "../Tree_Models"
tree_model_filename = f"{Dataset_name}_learned_tree.pkl"
tree_model_path = os.path.join(tree_model_dir, tree_model_filename)

# Check if the file exists
if os.path.exists(tree_model_path):
    with open(tree_model_path, "rb") as f:
        Tree = pickle.load(f)
    print(f"Successfully loaded tree model from {tree_model_path}")
else:
    print(f"Error: Tree model file not found at {tree_model_path}")

Successfully loaded tree model from ../Tree_Models\KentuckyDecisionTreeData_learned_tree.pkl


In [6]:
testing_start_time = time.time()
testaccu = Evaluate_model(Tree, Xtest, ytest)
testing_end_time = time.time()
print("Test accuracy for dataset {} is --- {}".format(Dataset_name, testaccu))
print("Testing time: {:.2f} seconds".format(testing_end_time - testing_start_time))

Test accuracy for dataset KentuckyDecisionTreeData is --- 0.7666666666666667
Testing time: 1.00 seconds


In [8]:
import torch
from fvcore.nn import FlopCountAnalysis, parameter_count_table
import numpy as np

def calculate_tree_flops(tree_model, sample_input_shape, T):
    """
    Calculate FLOPs for the entire NSTSC tree model.
    
    Args:
        tree_model: Dictionary containing the tree structure
        sample_input_shape: Shape of input data (N, 3*T)
        T: Number of time steps
    
    Returns:
        Total FLOPs for the tree model
    """
    total_flops = 0
    total_params = 0
    
    # Create sample input tensors for each view (raw, spectral, derivative)
    batch_size = 1  # Use batch size of 1 for FLOP calculation
    x1_sample = torch.randn(batch_size, T)  # Raw signal
    x2_sample = torch.randn(batch_size, T)  # Spectral features  
    x3_sample = torch.randn(batch_size, T)  # Derivative features
    
    print("Calculating FLOPs for NSTSC Tree Model")
    print("=" * 50)
    
    # Iterate through all nodes in the tree
    for node_idx, node in tree_model.items():
        if hasattr(node, 'bestmodel') and node.bestmodel is not None:
            try:
                # Calculate FLOPs for this node's model
                flops = FlopCountAnalysis(node.bestmodel, (x1_sample, x2_sample, x3_sample))
                node_flops = flops.total()
                total_flops += node_flops
                
                # Count parameters for this node's model  
                node_params = sum(p.numel() for p in node.bestmodel.parameters())
                total_params += node_params
                
                print(f"Node {node_idx}:")
                print(f"  Model Type: {type(node.bestmodel).__name__}")
                print(f"  FLOPs: {node_flops:,}")
                print(f"  Parameters: {node_params:,}")
                print(f"  Parameter Table:")
                print(parameter_count_table(node.bestmodel))
                print("-" * 30)
                
            except Exception as e:
                print(f"Warning: Could not calculate FLOPs for node {node_idx}: {e}")
    
    print(f"\nTotal Tree Model Statistics:")
    print(f"Total FLOPs: {total_flops:,}")
    print(f"Total Parameters: {total_params:,}")
    print(f"Number of nodes with models: {len([n for n in tree_model.values() if hasattr(n, 'bestmodel') and n.bestmodel is not None])}")
    
    return total_flops, total_params

# Calculate FLOPs for the loaded tree model
sample_shape = (N, 3*T)  # Shape based on your data (raw + spectral + derivative views)
total_flops, total_params = calculate_tree_flops(Tree, sample_shape, T)

# Additional analysis - FLOPs per inference
print(f"\nInference Analysis:")
print(f"FLOPs per sample classification: {total_flops:,}")
print(f"Average FLOPs per tree node: {total_flops / len(Tree):,.0f}")

# Memory footprint estimation
param_memory_mb = total_params * 4 / (1024 * 1024)  # Assuming float32 (4 bytes per param)
print(f"Estimated model memory: {param_memory_mb:.2f} MB")

Unsupported operator aten::mul encountered 7 time(s)
Unsupported operator aten::sub encountered 3 time(s)
Unsupported operator aten::sigmoid encountered 3 time(s)
Unsupported operator aten::softmax encountered 4 time(s)
Unsupported operator aten::rsub encountered 4 time(s)
Unsupported operator aten::sum encountered 4 time(s)
Unsupported operator aten::add encountered 4 time(s)
Unsupported operator aten::ones_like encountered 4 time(s)
Unsupported operator aten::min encountered 4 time(s)
Unsupported operator aten::sub encountered 3 time(s)
Unsupported operator aten::sigmoid encountered 3 time(s)
Unsupported operator aten::softmax encountered 4 time(s)
Unsupported operator aten::rsub encountered 4 time(s)
Unsupported operator aten::sum encountered 4 time(s)
Unsupported operator aten::add encountered 4 time(s)
Unsupported operator aten::ones_like encountered 4 time(s)
Unsupported operator aten::min encountered 4 time(s)


Calculating FLOPs for NSTSC Tree Model
Node 0:
  Model Type: TL_NN2
  FLOPs: 0
  Parameters: 11,059
  Parameter Table:
| name   | #elements or shape   |
|:-------|:---------------------|
| model  | 11.1K                |
|  t1    |  (1, 1228)           |
|  t2    |  (1, 1228)           |
|  t3    |  (1, 1228)           |
|  b1    |  (1, 1228)           |
|  b2    |  (1, 1228)           |
|  b3    |  (1, 1228)           |
|  A1    |  (1, 1228)           |
|  A2    |  (1, 1228)           |
|  A3    |  (1, 1228)           |
|  A4    |  (1, 3)              |
|  beta1 |  ()                  |
|  beta2 |  ()                  |
|  beta3 |  ()                  |
|  beta4 |  ()                  |
------------------------------


Unsupported operator aten::mul encountered 10 time(s)
Unsupported operator aten::sub encountered 7 time(s)
Unsupported operator aten::sigmoid encountered 3 time(s)
Unsupported operator aten::softmax encountered 7 time(s)
Unsupported operator aten::rsub encountered 7 time(s)
Unsupported operator aten::sum encountered 7 time(s)
Unsupported operator aten::add encountered 3 time(s)
Unsupported operator aten::min encountered 7 time(s)
Unsupported operator aten::ones_like encountered 6 time(s)
Unsupported operator aten::sub encountered 7 time(s)
Unsupported operator aten::sigmoid encountered 3 time(s)
Unsupported operator aten::softmax encountered 7 time(s)
Unsupported operator aten::rsub encountered 7 time(s)
Unsupported operator aten::sum encountered 7 time(s)
Unsupported operator aten::add encountered 3 time(s)
Unsupported operator aten::min encountered 7 time(s)
Unsupported operator aten::ones_like encountered 6 time(s)
Unsupported operator aten::where encountered 2 time(s)
Unsupported o

Node 2:
  Model Type: TL_NN5
  FLOPs: 0
  Parameters: 7,384
  Parameter Table:
| name     | #elements or shape   |
|:---------|:---------------------|
| model    | 7.4K                 |
|  t1      |  (1, 1)              |
|  b1      |  (1, 1)              |
|  A1_1    |  (1, 1, 1228)        |
|  beta1_1 |  ()                  |
|  beta1_2 |  ()                  |
|  A2_1    |  (1, 1228)           |
|  t2      |  (1, 1)              |
|  b2      |  (1, 1)              |
|  A1_2    |  (1, 1, 1228)        |
|  A2_2    |  (1, 1228)           |
|  beta2_1 |  ()                  |
|  beta2_2 |  ()                  |
|  t3      |  (1, 1)              |
|  b3      |  (1, 1)              |
|  A1_3    |  (1, 1, 1228)        |
|  A2_3    |  (1, 1228)           |
|  beta3_1 |  ()                  |
|  beta3_2 |  ()                  |
|  A4      |  (1, 3)              |
|  beta4   |  ()                  |
------------------------------
Node 3:
  Model Type: TL_NN1
  FLOPs: 0
  Parameters: 11,059
 

Unsupported operator aten::sub encountered 3 time(s)
Unsupported operator aten::sigmoid encountered 3 time(s)
Unsupported operator aten::softmax encountered 4 time(s)
Unsupported operator aten::rsub encountered 4 time(s)
Unsupported operator aten::sum encountered 4 time(s)
Unsupported operator aten::add encountered 4 time(s)
Unsupported operator aten::ones_like encountered 4 time(s)
Unsupported operator aten::min encountered 4 time(s)
Unsupported operator aten::mul encountered 7 time(s)
Unsupported operator aten::sigmoid encountered 3 time(s)
Unsupported operator aten::softmax encountered 4 time(s)
Unsupported operator aten::rsub encountered 4 time(s)
Unsupported operator aten::sum encountered 4 time(s)
Unsupported operator aten::add encountered 4 time(s)
Unsupported operator aten::ones_like encountered 4 time(s)
Unsupported operator aten::min encountered 4 time(s)
Unsupported operator aten::mul encountered 7 time(s)
Unsupported operator aten::sub encountered 7 time(s)
Unsupported oper

Node 13:
  Model Type: TL_NN2
  FLOPs: 0
  Parameters: 11,059
  Parameter Table:
| name   | #elements or shape   |
|:-------|:---------------------|
| model  | 11.1K                |
|  t1    |  (1, 1228)           |
|  t2    |  (1, 1228)           |
|  t3    |  (1, 1228)           |
|  b1    |  (1, 1228)           |
|  b2    |  (1, 1228)           |
|  b3    |  (1, 1228)           |
|  A1    |  (1, 1228)           |
|  A2    |  (1, 1228)           |
|  A3    |  (1, 1228)           |
|  A4    |  (1, 3)              |
|  beta1 |  ()                  |
|  beta2 |  ()                  |
|  beta3 |  ()                  |
|  beta4 |  ()                  |
------------------------------
Node 15:
  Model Type: TL_NN1
  FLOPs: 0
  Parameters: 11,059
  Parameter Table:
| name   | #elements or shape   |
|:-------|:---------------------|
| model  | 11.1K                |
|  t1    |  (1, 1228)           |
|  t2    |  (1, 1228)           |
|  t3    |  (1, 1228)           |
|  b1    |  (1, 1228)    

Unsupported operator aten::sub encountered 7 time(s)
Unsupported operator aten::sigmoid encountered 3 time(s)
Unsupported operator aten::softmax encountered 4 time(s)
Unsupported operator aten::rsub encountered 4 time(s)
Unsupported operator aten::sum encountered 4 time(s)
Unsupported operator aten::ones_like encountered 4 time(s)
Unsupported operator aten::min encountered 4 time(s)
Unsupported operator aten::sigmoid encountered 3 time(s)
Unsupported operator aten::softmax encountered 4 time(s)
Unsupported operator aten::rsub encountered 4 time(s)
Unsupported operator aten::sum encountered 4 time(s)
Unsupported operator aten::ones_like encountered 4 time(s)
Unsupported operator aten::min encountered 4 time(s)
Unsupported operator aten::mul encountered 7 time(s)
Unsupported operator aten::sub encountered 7 time(s)
Unsupported operator aten::sigmoid encountered 3 time(s)
Unsupported operator aten::softmax encountered 4 time(s)
Unsupported operator aten::rsub encountered 4 time(s)
Unsuppo

Node 34:
  Model Type: TL_NN1
  FLOPs: 0
  Parameters: 11,059
  Parameter Table:
| name   | #elements or shape   |
|:-------|:---------------------|
| model  | 11.1K                |
|  t1    |  (1, 1228)           |
|  t2    |  (1, 1228)           |
|  t3    |  (1, 1228)           |
|  b1    |  (1, 1228)           |
|  b2    |  (1, 1228)           |
|  b3    |  (1, 1228)           |
|  A1    |  (1, 1228)           |
|  A2    |  (1, 1228)           |
|  A3    |  (1, 1228)           |
|  A4    |  (1, 3)              |
|  beta1 |  ()                  |
|  beta2 |  ()                  |
|  beta3 |  ()                  |
|  beta4 |  ()                  |
------------------------------
Node 37:
  Model Type: TL_NN1
  FLOPs: 0
  Parameters: 11,059
  Parameter Table:
| name   | #elements or shape   |
|:-------|:---------------------|
| model  | 11.1K                |
|  t1    |  (1, 1228)           |
|  t2    |  (1, 1228)           |
|  t3    |  (1, 1228)           |
|  b1    |  (1, 1228)    

Unsupported operator aten::sub encountered 7 time(s)
Unsupported operator aten::sigmoid encountered 3 time(s)
Unsupported operator aten::sigmoid encountered 3 time(s)
Unsupported operator aten::softmax encountered 4 time(s)
Unsupported operator aten::rsub encountered 4 time(s)
Unsupported operator aten::sum encountered 4 time(s)
Unsupported operator aten::ones_like encountered 4 time(s)
Unsupported operator aten::min encountered 4 time(s)
Unsupported operator aten::softmax encountered 4 time(s)
Unsupported operator aten::rsub encountered 4 time(s)
Unsupported operator aten::sum encountered 4 time(s)
Unsupported operator aten::ones_like encountered 4 time(s)
Unsupported operator aten::min encountered 4 time(s)
Unsupported operator aten::mul encountered 7 time(s)
Unsupported operator aten::sub encountered 7 time(s)
Unsupported operator aten::sigmoid encountered 3 time(s)
Unsupported operator aten::softmax encountered 4 time(s)
Unsupported operator aten::rsub encountered 4 time(s)
Unsuppo

Node 50:
  Model Type: TL_NN1
  FLOPs: 0
  Parameters: 11,059
  Parameter Table:
| name   | #elements or shape   |
|:-------|:---------------------|
| model  | 11.1K                |
|  t1    |  (1, 1228)           |
|  t2    |  (1, 1228)           |
|  t3    |  (1, 1228)           |
|  b1    |  (1, 1228)           |
|  b2    |  (1, 1228)           |
|  b3    |  (1, 1228)           |
|  A1    |  (1, 1228)           |
|  A2    |  (1, 1228)           |
|  A3    |  (1, 1228)           |
|  A4    |  (1, 3)              |
|  beta1 |  ()                  |
|  beta2 |  ()                  |
|  beta3 |  ()                  |
|  beta4 |  ()                  |
------------------------------
Node 51:
  Model Type: TL_NN3
  FLOPs: 0
  Parameters: 3,697
  Parameter Table:
| name   | #elements or shape   |
|:-------|:---------------------|
| model  | 3.7K                 |
|  t1    |  (1, 1)              |
|  t2    |  (1, 1)              |
|  t3    |  (1, 1)              |
|  b1    |  (1, 1)        

In [9]:
def calculate_manual_flops(tree_model, T):
    """
    Manually calculate FLOPs for NSTSC tree models by counting operations.
    This accounts for the specific operations in TL_NN models that fvcore doesn't recognize.
    
    Args:
        tree_model: Dictionary containing the tree structure
        T: Number of time steps
    
    Returns:
        Total FLOPs for manual calculation
    """
    
    def count_tlnn_flops(model_type, T):
        """Count FLOPs for different TL_NN model types"""
        
        if model_type in ['TL_NN1', 'TL_NN2']:
            # Operations for TL_NN1 and TL_NN2:
            # 1. Element-wise multiplication: x * t (3 times, each T operations)
            mult_ops = 3 * T
            
            # 2. Element-wise subtraction: result - b (3 times, each T operations)  
            sub_ops = 3 * T
            
            # 3. Sigmoid activation (3 times, each T operations, ~4 ops per sigmoid)
            sigmoid_ops = 3 * T * 4
            
            # 4. Softmax operations (4 times: 3 over T dimensions, 1 over 3 dimensions)
            # Softmax ≈ exp + sum + div = ~3 ops per element
            softmax_ops = (3 * T * 3) + (3 * 3)
            
            # 5. Element-wise operations for weighted bias computation
            # (1-sigmoid) or sigmoid, then multiply by A_sm, then sum
            weighted_ops = 3 * (T * 2 + T + 1)  # (1-x), multiply, sum for each of 3 components
            
            # 6. Final weighted combination (3 elements)
            final_ops = 3 * 3 + 1  # multiply by weights + sum
            
            # 7. Clamp operation (min/max comparisons)
            clamp_ops = 4 * 2  # 4 clamp calls, 2 ops each (min and max)
            
            total = mult_ops + sub_ops + sigmoid_ops + softmax_ops + weighted_ops + final_ops + clamp_ops
            
        elif model_type == 'TL_NN3':
            # TL_NN3 has similar structure but with t parameters of size 1 instead of T
            # Most operations are similar but with reduced dimensionality for t parameters
            mult_ops = 3 * 1  # t parameters are size 1
            sub_ops = 3 * 1
            sigmoid_ops = 3 * 1 * 4
            
            # Softmax still operates over T and 3 dimensions
            softmax_ops = (3 * T * 3) + (3 * 3)
            
            # Weighted operations still use A parameters of size T
            weighted_ops = 3 * (T * 2 + T + 1)
            final_ops = 3 * 3 + 1
            clamp_ops = 4 * 2
            
            total = mult_ops + sub_ops + sigmoid_ops + softmax_ops + weighted_ops + final_ops + clamp_ops
            
        elif model_type in ['TL_NN4', 'TL_NN5', 'TL_NN6']:
            # Similar structure to TL_NN1/TL_NN2, use same calculation
            total = count_tlnn_flops('TL_NN1', T)
        else:
            total = 0
            
        return total
    
    total_manual_flops = 0
    flop_breakdown = {}
    
    print("Manual FLOP Calculation for NSTSC Tree Model")
    print("=" * 60)
    
    # Count FLOPs for each node
    for node_idx, node in tree_model.items():
        if hasattr(node, 'bestmodel') and node.bestmodel is not None:
            model_type = type(node.bestmodel).__name__
            node_flops = count_tlnn_flops(model_type, T)
            total_manual_flops += node_flops
            
            if model_type not in flop_breakdown:
                flop_breakdown[model_type] = {'count': 0, 'flops_per_model': node_flops, 'total_flops': 0}
            
            flop_breakdown[model_type]['count'] += 1
            flop_breakdown[model_type]['total_flops'] += node_flops
            
            print(f"Node {node_idx}: {model_type} - {node_flops:,} FLOPs")
    
    print("\n" + "=" * 60)
    print("FLOP Breakdown by Model Type:")
    print("-" * 40)
    
    for model_type, stats in flop_breakdown.items():
        print(f"{model_type}:")
        print(f"  Count: {stats['count']} models")
        print(f"  FLOPs per model: {stats['flops_per_model']:,}")
        print(f"  Total FLOPs: {stats['total_flops']:,}")
        print()
    
    print(f"Total Manual FLOPs: {total_manual_flops:,}")
    print(f"Average FLOPs per model: {total_manual_flops / len([n for n in tree_model.values() if hasattr(n, 'bestmodel') and n.bestmodel is not None]):,.0f}")
    
    return total_manual_flops, flop_breakdown

# Calculate manual FLOPs
manual_flops, breakdown = calculate_manual_flops(Tree, T)

# Comparison with automatic calculation
print(f"\n" + "=" * 60)
print("FLOP Calculation Comparison:")
print(f"fvcore (automatic): {total_flops:,} FLOPs")
print(f"Manual calculation: {manual_flops:,} FLOPs")
print(f"Difference: {abs(manual_flops - total_flops):,} FLOPs")

# Performance metrics
print(f"\nModel Efficiency Metrics:")
print(f"FLOPs per parameter: {manual_flops / total_params:.2f}")
print(f"Parameters per FLOP: {total_params / manual_flops:.6f}")

# Test dataset inference cost
test_samples = len(ytest)
total_inference_flops = manual_flops * test_samples
print(f"\nInference on test set ({test_samples} samples):")
print(f"Total FLOPs: {total_inference_flops:,}")
print(f"Average FLOPs per sample: {manual_flops:,}")

Manual FLOP Calculation for NSTSC Tree Model
Node 0: TL_NN2 - 44,238 FLOPs
Node 2: TL_NN5 - 44,238 FLOPs
Node 3: TL_NN1 - 44,238 FLOPs
Node 4: TL_NN1 - 44,238 FLOPs
Node 5: TL_NN1 - 44,238 FLOPs
Node 6: TL_NN1 - 44,238 FLOPs
Node 7: TL_NN2 - 44,238 FLOPs
Node 8: TL_NN4 - 44,238 FLOPs
Node 9: TL_NN1 - 44,238 FLOPs
Node 11: TL_NN1 - 44,238 FLOPs
Node 13: TL_NN2 - 44,238 FLOPs
Node 15: TL_NN1 - 44,238 FLOPs
Node 16: TL_NN1 - 44,238 FLOPs
Node 17: TL_NN1 - 44,238 FLOPs
Node 21: TL_NN4 - 44,238 FLOPs
Node 22: TL_NN1 - 44,238 FLOPs
Node 24: TL_NN2 - 44,238 FLOPs
Node 25: TL_NN1 - 44,238 FLOPs
Node 26: TL_NN4 - 44,238 FLOPs
Node 28: TL_NN1 - 44,238 FLOPs
Node 31: TL_NN1 - 44,238 FLOPs
Node 33: TL_NN1 - 44,238 FLOPs
Node 34: TL_NN1 - 44,238 FLOPs
Node 37: TL_NN1 - 44,238 FLOPs
Node 38: TL_NN1 - 44,238 FLOPs
Node 39: TL_NN1 - 44,238 FLOPs
Node 40: TL_NN2 - 44,238 FLOPs
Node 41: TL_NN1 - 44,238 FLOPs
Node 44: TL_NN1 - 44,238 FLOPs
Node 45: TL_NN1 - 44,238 FLOPs
Node 46: TL_NN1 - 44,238 FLOPs
Nod

In [10]:
# Comprehensive Performance Summary
print("🔍 NSTSC Tree Model Performance Profile")
print("=" * 80)

# Model Architecture Summary
print(f"📊 Model Architecture:")
print(f"   Dataset: {Dataset_name}")
print(f"   Time series length (T): {T}")
print(f"   Number of samples (N): {N}")
print(f"   Input dimensions: 3 × {T} (raw + spectral + derivative)")
print(f"   Tree nodes: {len(Tree)}")
print(f"   Active model nodes: {len([n for n in Tree.values() if hasattr(n, 'bestmodel') and n.bestmodel is not None])}")

# Computational Complexity
print(f"\n⚡ Computational Complexity:")
print(f"   Total Parameters: {total_params:,}")
print(f"   Manual FLOPs: {manual_flops:,}")
print(f"   Memory footprint: {param_memory_mb:.2f} MB")
print(f"   FLOPs per parameter: {manual_flops / total_params:.2f}")

# Performance Metrics
print(f"\n🎯 Performance Metrics:")
print(f"   Test Accuracy: {testaccu:.4f}")
print(f"   Testing Time: {testing_end_time - testing_start_time:.2f} seconds")
print(f"   Throughput: {len(ytest) / (testing_end_time - testing_start_time):.1f} samples/second")

# Efficiency Analysis
flops_per_second = manual_flops * len(ytest) / (testing_end_time - testing_start_time)
print(f"\n🚀 Efficiency Analysis:")
print(f"   FLOPs per inference: {manual_flops:,}")
print(f"   Computational throughput: {flops_per_second:,.0f} FLOPs/second")
print(f"   Accuracy per FLOP: {testaccu / manual_flops:.10f}")

# Save profiling results
profiling_results = {
    'dataset_name': Dataset_name,
    'model_architecture': {
        'time_series_length': T,
        'num_samples': N,
        'tree_nodes': len(Tree),
        'active_nodes': len([n for n in Tree.values() if hasattr(n, 'bestmodel') and n.bestmodel is not None])
    },
    'computational_metrics': {
        'total_parameters': total_params,
        'manual_flops': manual_flops,
        'memory_mb': param_memory_mb,
        'flops_per_parameter': manual_flops / total_params
    },
    'performance_metrics': {
        'test_accuracy': testaccu,
        'testing_time_seconds': testing_end_time - testing_start_time,
        'throughput_samples_per_second': len(ytest) / (testing_end_time - testing_start_time),
        'flops_per_inference': manual_flops,
        'computational_throughput_flops_per_second': flops_per_second,
        'accuracy_per_flop': testaccu / manual_flops
    },
    'model_breakdown': breakdown
}

# Save to file
import json
profile_output_path = f"../Tree_Models/{Dataset_name}_performance_profile.json"
with open(profile_output_path, 'w') as f:
    # Convert numpy types to Python types for JSON serialization
    def convert_numpy(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj
    
    # Convert all numpy types in the dictionary
    def clean_dict(d):
        if isinstance(d, dict):
            return {k: clean_dict(v) for k, v in d.items()}
        elif isinstance(d, list):
            return [clean_dict(v) for v in d]
        else:
            return convert_numpy(d)
    
    json.dump(clean_dict(profiling_results), f, indent=2)

print(f"\n💾 Profiling results saved to: {profile_output_path}")
print("=" * 80)

🔍 NSTSC Tree Model Performance Profile
📊 Model Architecture:
   Dataset: KentuckyDecisionTreeData
   Time series length (T): 1228
   Number of samples (N): 150
   Input dimensions: 3 × 1228 (raw + spectral + derivative)
   Tree nodes: 54
   Active model nodes: 36

⚡ Computational Complexity:
   Total Parameters: 365,001
   Manual FLOPs: 1,570,482
   Memory footprint: 1.39 MB
   FLOPs per parameter: 4.30

🎯 Performance Metrics:
   Test Accuracy: 0.7667
   Testing Time: 1.00 seconds
   Throughput: 29.9 samples/second

🚀 Efficiency Analysis:
   FLOPs per inference: 1,570,482
   Computational throughput: 46,994,819 FLOPs/second
   Accuracy per FLOP: 0.0000004882

💾 Profiling results saved to: ../Tree_Models/KentuckyDecisionTreeData_performance_profile.json
